# 🌱 VitaLlama 3.1 — Clinical Nutritionist LLM Server
### Production FastAPI + 3-Tier Hybrid Memory Engine on Google Colab (Tesla T4)

This notebook turns your **fine-tuned Llama 3.1 8B Clinical Nutritionist** model into a
public streaming inference API.  

**Architecture highlights:**
| Tier | Component | Latency |
|------|-----------|----------|
| 1 | SHA-256 Query Cache (`TTLCache`) | < 10 ms |
| 2 | Sliding Window (K = 6 raw turns) | — |
| 3 | Rolling Clinical Summary (async LLM) | Background |

Run cells **1 → 6** in order. Paste the printed public URL into `index.html`.

In [ ]:
a=4
b=5
a+b

In [1]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — GPU Verification & Google Drive Mount                 ║
# ╚═══════════════════════════════════════════════════════════════════╝
import os, torch
from google.colab import drive

print("🔍  Checking GPU accelerator…")
if not torch.cuda.is_available():
    raise SystemError(
        "❌  No GPU detected!  Go to  Runtime → Change runtime type → T4 GPU."
    )

gpu_name  = torch.cuda.get_device_name(0)
vram_gb   = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
print(f"✅  GPU : {gpu_name}  ({vram_gb} GB VRAM)")

print("\n📁  Mounting Google Drive…")
drive.mount("/content/drive")

# ----- Resolve model path automatically -----
MODEL_DRIVE_PATH   = "/content/drive/MyDrive/Nutritionist_LLM_Phase1/merged_model_drive"
CHECKPOINT_DIR     = "/content/drive/MyDrive/Nutritionist_LLM_Phase1/checkpoints"
FALLBACK_HF_MODEL  = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"

if os.path.isdir(MODEL_DRIVE_PATH):
    SELECTED_MODEL = MODEL_DRIVE_PATH
    print(f"✅  Merged model found : {MODEL_DRIVE_PATH}")
elif os.path.isdir(CHECKPOINT_DIR):
    SELECTED_MODEL = CHECKPOINT_DIR
    print(f"⚠️   Merged model missing — using checkpoints : {CHECKPOINT_DIR}")
else:
    SELECTED_MODEL = FALLBACK_HF_MODEL
    print(f"ℹ️   No local weights — falling back to HF base : {FALLBACK_HF_MODEL}")

🔍  Checking GPU accelerator…
✅  GPU : Tesla T4  (14.56 GB VRAM)

📁  Mounting Google Drive…
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅  Merged model found : /content/drive/MyDrive/Nutritionist_LLM_Phase1/merged_model_drive


In [2]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Install Runtime Dependencies                         ║
# ╚═══════════════════════════════════════════════════════════════════╝
print("⏳  Installing Unsloth, FastAPI, SSE tooling, and tunnel packages…")

!pip uninstall -y -q xformers 2>/dev/null
!pip install -q --upgrade --no-cache-dir \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets
!pip install -q fastapi "uvicorn[standard]" pyngrok cachetools pydantic nest-asyncio

print("✅  All packages installed!")

⏳  Installing Unsloth, FastAPI, SSE tooling, and tunnel packages…
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅  All packages installed!


In [3]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Load Model + Bind ChatML Template                    ║
# ╚═══════════════════════════════════════════════════════════════════╝
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MAX_SEQ_LENGTH = 2048

print(f"⏳  Loading model from: {SELECTED_MODEL} …")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = SELECTED_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,       # auto-selects fp16 on T4
    load_in_4bit   = True,       # NF4 quantisation → ~5 GB VRAM
)

# Enforce ChatML delimiters: <|im_start|> / <|im_end|>
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {
        "role": "role", "content": "content",
        "user": "user", "assistant": "assistant",
    },
)

# Engage Unsloth 2× faster inference kernels
FastLanguageModel.for_inference(model)

alloc = round(torch.cuda.memory_allocated(0) / 1024**3, 2)
resv  = round(torch.cuda.memory_reserved(0)  / 1024**3, 2)
print(f"✅  Model ready!  VRAM  allocated {alloc} GB  |  reserved {resv} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳  Loading model from: /content/drive/MyDrive/Nutritionist_LLM_Phase1/merged_model_drive …
==((====))==  Unsloth 2026.10.11: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/Nutritionist_LLM_Phase1/merged_model_drive as a legacy tokenizer.
Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_opr4_ny4/tokenizer_config.json.


✅  Model ready!  VRAM  allocated 5.32 GB  |  reserved 5.46 GB


In [4]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Three-Tier Hybrid Memory Engine                      ║
# ╚═══════════════════════════════════════════════════════════════════╝
import re, time, json, uuid, hashlib, logging
from typing import List, Dict, Optional
from cachetools import TTLCache

logging.basicConfig(level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("VitaLlama")

SLIDING_WINDOW_K = 6   # 3 user + 3 assistant turns kept verbatim

# ── Tier 1: Exact / Normalized Query Cache ─────────────────────────
# 4-hour TTL, 1 024 slots.  sha256(normalised_query) → cached answer
tier1_cache: TTLCache = TTLCache(maxsize=1024, ttl=3600 * 4)
cache_stats = dict(total=0, hits=0, misses=0)

def _normalise(text: str) -> str:
    """Lower-case, strip punctuation & collapse whitespace."""
    t = text.lower().strip()
    t = re.sub(r"[\r\n\t]+", " ", t)
    t = re.sub(r"[^\w\s]", "", t)
    return re.sub(r"\s+", " ", t).strip()

def _hash(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()

# ── Tier 2 + 3: Session Memory Object ──────────────────────────────
class SessionMemory:
    """Per-session state for Tier 2 (sliding window) + Tier 3 (clinical summary)."""
    def __init__(self, sid: str):
        self.sid            = sid
        self.history: List[Dict[str, str]] = []
        self.clinical_summary = "No prior clinical history recorded."
        self.is_summarising   = False
        self.last_active      = time.time()

    def add(self, role: str, content: str):
        self.history.append({"role": role, "content": content})
        self.last_active = time.time()

    def window(self, k=SLIDING_WINDOW_K):
        return self.history[-k:] if len(self.history) > k else list(self.history)

    def overflow(self, k=SLIDING_WINDOW_K):
        return self.history[:-k] if len(self.history) > k else []

    def reset(self):
        self.history.clear()
        self.clinical_summary = "No prior clinical history recorded."

sessions: Dict[str, SessionMemory] = {}

def get_session(sid: Optional[str] = None) -> SessionMemory:
    sid = sid or str(uuid.uuid4())
    if sid not in sessions:
        sessions[sid] = SessionMemory(sid)
    return sessions[sid]

# ── Prompt Construction (ChatML with dynamic clinical summary) ─────
BASE_SYSTEM = (
    "You are an empathetic, evidence-based clinical nutritionist. "
    "Provide scientifically accurate dietary, macronutrient, and "
    "micronutrient guidance. Never diagnose severe medical conditions "
    "(e.g., CKD, Type 1 Diabetes); always refer such patients to a physician."
)

SUMMARISER_TEMPLATE = (
    "<|im_start|>system\n"
    "You are a clinical data extraction assistant. Update the cumulative "
    "patient profile from the new dialogue turns below.\n"
    "Focus on: Demographics, Anthropometrics & Goals, Medical Conditions "
    "& GI Health, Dietary Preferences, Supplements & Hydration.\n\n"
    "Existing profile:\n{existing}\n\n"
    "New turns:\n{turns}\n\n"
    "Output ONLY the updated profile in 3-5 concise bullet points."
    "<|im_end|>\n<|im_start|>assistant\n"
)

def build_prompt(session: SessionMemory, user_msg: str) -> str:
    sys_block = (
        f"{BASE_SYSTEM}\n\n"
        f"[PATIENT CLINICAL PROFILE]\n{session.clinical_summary}\n"
        f"Personalise all guidance to the profile above."
    )
    parts = [f"<|im_start|>system\n{sys_block}<|im_end|>"]
    for m in session.window(SLIDING_WINDOW_K):
        parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>")
    parts.append(f"<|im_start|>user\n{user_msg}<|im_end|>")
    parts.append("<|im_start|>assistant\n")
    return "\n".join(parts)

print("✅  3-Tier Memory Engine initialised (K =", SLIDING_WINDOW_K, "turns)")

✅  3-Tier Memory Engine initialised (K = 6 turns)


In [5]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — FastAPI App + Non-Blocking SSE Token Streaming        ║
# ╚═══════════════════════════════════════════════════════════════════╝
import asyncio, nest_asyncio
nest_asyncio.apply()          # Allow asyncio inside Jupyter's event loop

from threading import Thread, Lock
from pydantic import BaseModel, Field
from fastapi import FastAPI, BackgroundTasks, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse, JSONResponse
from transformers import TextIteratorStreamer

gpu_lock = Lock()             # serialise single-GPU generation

app = FastAPI(
    title="VitaLlama 3.1 — Clinical Nutritionist API",
    version="1.0.0",
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_credentials=True,
    allow_methods=["*"], allow_headers=["*"],
    expose_headers=["*"],
)

# ── Request / Response Schemas ─────────────────────────────────────
class ChatRequest(BaseModel):
    message:     str            = Field(..., min_length=1, max_length=4000)
    session_id:  Optional[str]  = None
    temperature: Optional[float]= 0.3
    max_tokens:  Optional[int]  = 512

# ── Health / Diagnostics ───────────────────────────────────────────
@app.get("/")
@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "model_loaded": True,
        "sessions": len(sessions),
        "cache_size": len(tier1_cache),
        "cache_stats": cache_stats,
        "vram_alloc_gb": round(torch.cuda.memory_allocated(0)/1024**3, 2),
    }

# ── Session Inspector ──────────────────────────────────────────────
@app.get("/session/{sid}")
async def inspect_session(sid: str):
    if sid not in sessions:
        raise HTTPException(404, "Session not found")
    s = sessions[sid]
    return {
        "session_id":       s.sid,
        "clinical_summary": s.clinical_summary,
        "total_messages":   len(s.history),
        "window_active":    len(s.window()),
        "is_summarising":   s.is_summarising,
        "history":          s.history,
    }

@app.post("/session/{sid}/reset")
async def reset_session(sid: str):
    if sid in sessions:
        sessions[sid].reset()
    return {"status": "ok", "message": "Session memory cleared."}

@app.get("/cache/stats")
async def cache_analytics():
    ratio = round(cache_stats["hits"] / max(cache_stats["total"], 1) * 100, 2)
    return {**cache_stats, "entries": len(tier1_cache), "hit_pct": ratio}

# ── Tier 3 Background Summariser ───────────────────────────────────
def _run_tier3(session: SessionMemory, overflow: List[Dict[str, str]]):
    """GPU-based background summarisation of old turns into clinical profile."""
    if not overflow or session.is_summarising:
        return
    session.is_summarising = True
    try:
        turns_txt = "\n".join(
            f"{t['role'].upper()}: {t['content']}" for t in overflow
        )
        prompt = SUMMARISER_TEMPLATE.format(
            existing=session.clinical_summary, turns=turns_txt
        )
        ids = tokenizer(prompt, return_tensors="pt").to("cuda")
        with gpu_lock, torch.inference_mode():
            out = model.generate(
                **ids, max_new_tokens=200,
                temperature=0.15, use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = out[0][ids.input_ids.shape[1]:]
        text = tokenizer.decode(gen, skip_special_tokens=True)
        text = text.replace("<|im_end|>", "").strip()
        if text:
            session.clinical_summary = text
            log.info(f"[TIER 3] Session {session.sid[:8]}… summary updated.")
    except Exception as e:
        log.error(f"Tier 3 error: {e}")
    finally:
        session.is_summarising = False

# ── Main Chat SSE Endpoint ─────────────────────────────────────────
@app.post("/chat")
async def chat(req: ChatRequest, bg: BackgroundTasks):
    t0 = time.time()
    cache_stats["total"] += 1
    session   = get_session(req.session_id)
    user_msg  = req.message.strip()
    norm      = _normalise(user_msg)
    qhash     = _hash(norm)

    sse_headers = {
        "Cache-Control": "no-cache",
        "Connection":    "keep-alive",
        "X-Accel-Buffering": "no",
        "ngrok-skip-browser-warning": "true",
    }

    # ──────── TIER 1: Instant Cache Hit ────────
    if qhash in tier1_cache:
        cache_stats["hits"] += 1
        cached = tier1_cache[qhash]
        cached["hit_count"] += 1
        answer = cached["response"]
        lat = round((time.time() - t0) * 1000, 2)

        session.add("user", user_msg)
        session.add("assistant", answer)
        log.info(f"⚡ CACHE HIT  {qhash[:10]}  {lat}ms")

        async def _cached():
            yield (
                f"event: metadata\n"
                f"data: {json.dumps(dict(source='cache', cache_hit=True, latency_ms=lat, session_id=session.sid, clinical_summary=session.clinical_summary, window_turns=len(session.window())))}\n\n"
            )
            words = answer.split(" ")
            buf = []
            for i, w in enumerate(words):
                buf.append(w)
                if len(buf) >= 5 or i == len(words) - 1:
                    chunk = " ".join(buf) + (" " if i < len(words)-1 else "")
                    yield f"event: token\ndata: {json.dumps(dict(token=chunk))}\n\n"
                    buf = []
                    await asyncio.sleep(0.006)
            yield f"event: done\ndata: {json.dumps(dict(done=True, source='cache', latency_ms=lat))}\n\n"

        return StreamingResponse(_cached(), media_type="text/event-stream", headers=sse_headers)

    # ──────── TIER 2 / 3: Live GPU Inference ────────
    cache_stats["misses"] += 1
    prompt = build_prompt(session, user_msg)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    streamer = TextIteratorStreamer(
        tokenizer, skip_prompt=True, skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )
    gen_kw = dict(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        streamer=streamer,
        max_new_tokens=req.max_tokens,
        temperature=max(req.temperature, 0.01),
        top_p=0.9, repetition_penalty=1.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    def _worker():
        with gpu_lock, torch.inference_mode():
            model.generate(**gen_kw)

    Thread(target=_worker, daemon=True).start()

    async def _gpu():
        loop = asyncio.get_running_loop()
        parts = []
        first_tok = None

        yield (
            f"event: metadata\n"
            f"data: {json.dumps(dict(source='gpu', cache_hit=False, session_id=session.sid, clinical_summary=session.clinical_summary, window_turns=len(session.window())+1))}\n\n"
        )

        while True:
            tok = await loop.run_in_executor(None, lambda: next(streamer, None))
            if tok is None:
                break
            if first_tok is None:
                first_tok = time.time()
                ttft = round((first_tok - t0)*1000, 2)
                yield f"event: ttft\ndata: {json.dumps(dict(ttft_ms=ttft))}\n\n"

            clean = tok.replace("<|im_end|>", "").replace("<|end_of_text|>", "")
            if clean:
                parts.append(clean)
                yield f"event: token\ndata: {json.dumps(dict(token=clean))}\n\n"

        full = "".join(parts).strip()
        dur  = round((time.time() - t0)*1000, 2)

        session.add("user", user_msg)
        session.add("assistant", full)

        # Tier 1 — cache new answer
        tier1_cache[qhash] = dict(response=full, ts=time.time(), hit_count=0)

        # Tier 3 — trigger background summariser when overflow ≥ 2
        ovf = session.overflow(SLIDING_WINDOW_K)
        if len(ovf) >= 2 and not session.is_summarising:
            log.info(f"🔄 Tier 3 summariser queued for session {session.sid[:8]}…")
            bg.add_task(_run_tier3, session, ovf)

        yield (
            f"event: done\n"
            f"data: {json.dumps(dict(done=True, source='gpu', duration_ms=dur, clinical_summary=session.clinical_summary))}\n\n"
        )

    return StreamingResponse(_gpu(), media_type="text/event-stream", headers=sse_headers)

print("✅  FastAPI app ready with /chat, /health, /session, /cache endpoints!")

✅  FastAPI app ready with /chat, /health, /session, /cache endpoints!


In [6]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Launch Uvicorn + Public Tunnel (Python 3.13 Fixed)    ║
# ╚═══════════════════════════════════════════════════════════════════╝
import asyncio
import time
import uvicorn
from threading import Thread

PORT = 8000

# (Optional) Enter your ngrok auth token if using ngrok
NGROK_AUTH_TOKEN = ""   # ← paste your token here, or leave blank to use Cloudflare (Cell 7)

# 1. Kill any prior stuck instances on port 8000
!fuser -k 8000/tcp > /dev/null 2>&1

# 2. Configure Uvicorn server
config = uvicorn.Config(app=app, host="0.0.0.0", port=PORT, log_level="warning")
server = uvicorn.Server(config)

# 3. Dedicated thread runner with its own event loop (bypasses Python 3.13 loop_factory bug)
def start_uvicorn():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(server.serve())

server_thread = Thread(target=start_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)

# 4. Connect Ngrok tunnel (if token provided)
public_url = None
if NGROK_AUTH_TOKEN.strip():
    try:
        from pyngrok import ngrok
        ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
        tunnel = ngrok.connect(PORT)
        public_url = tunnel.public_url
    except Exception as e:
        print(f"⚠️ Ngrok error: {e}")

print("\n" + "═" * 68)
print("  🚀 VITALLLAMA 3.1 — CLINICAL NUTRITIONIST API IS LIVE!")
print("═" * 68)
print(f"  Local  : http://localhost:{PORT}")
if public_url:
    print(f"  Public : {public_url}")
    print(f"  👉 Copy this URL into index.html → 'Colab API URL' input box!")
else:
    print("  ℹ️ No Ngrok token provided. Run Cell 7 to launch Cloudflare Tunnel.")
print("═" * 68)


════════════════════════════════════════════════════════════════════
  🚀 VITALLLAMA 3.1 — CLINICAL NUTRITIONIST API IS LIVE!
════════════════════════════════════════════════════════════════════
  Local  : http://localhost:8000
  ℹ️ No Ngrok token provided. Run Cell 7 to launch Cloudflare Tunnel.
════════════════════════════════════════════════════════════════════


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 (OPTIONAL) — Free Cloudflare Tunnel (no account)       ║
# ╚═══════════════════════════════════════════════════════════════════╝
# Run this cell ONLY if you skipped Ngrok above.

print("⏳  Installing Cloudflare Tunnel…")
!wget -qnc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess, re as _re

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print("🔍  Waiting for Cloudflare URL…")
for line in proc.stdout:
    if "trycloudflare.com" in line:
        m = _re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if m:
            cf_url = m.group(0)
            print("\n" + "═"*68)
            print("  🌐  CLOUDFLARE TUNNEL ACTIVE")
            print(f"  Public :  {cf_url}")
            print("  👉  Paste into index.html → 'Colab API URL' box!")
            print("═"*68)
            break

⏳  Installing Cloudflare Tunnel…
🔍  Waiting for Cloudflare URL…

════════════════════════════════════════════════════════════════════
  🌐  CLOUDFLARE TUNNEL ACTIVE
  Public :  https://prepare-information-drive-truck.trycloudflare.com
  👉  Paste into index.html → 'Colab API URL' box!
════════════════════════════════════════════════════════════════════


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tok

In [ ]:
import requests
try:
    res = requests.get("http://localhost:8000/health")
    print("✅ Local Server Status:", res.json())
except Exception as e:
    print("❌ Local Server is DOWN:", e)